# Belle II Continuum Suppression

Binary signal-versus-continuum classification using the instructor's Belle II CSV. The private data is not committed.

## 1. Setup and reproducibility

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.baselines import evaluate_baselines
from src.config import (
    DB_PATH, RANDOM_STATE, RAW_DATA_DIR, TEST_SIZE, VALIDATION_SIZE
)
from src.data_io import (
    assign_split, audit_frame, load_raw_to_db, prepare_professor_binary
)
from src.db import read_table, table_exists, update_cell, write_table
from src.eda import (class_balance, feature_columns, feature_correlations,
                     feature_separation, plot_class_balance,
                     plot_correlation_heatmap, plot_feature_distributions,
                     training_events)
from src.models import fit_ankit_validation, fit_vishruti_validation

print(f"Project root: {PROJECT_ROOT}")
print(
    f"Random state: {RANDOM_STATE}; split fractions: "
    f"train={1 - VALIDATION_SIZE - TEST_SIZE:.0%}, "
    f"validation={VALIDATION_SIZE:.0%}, test={TEST_SIZE:.0%}"
)

## 2. Instructor dataset

Use `data_hep - data_hep.csv` only. Its process flags are converted to the binary target below.

The original CSV remains private and unchanged.

## 3. Locate the instructor CSV

In [ ]:
raw_csv_path = RAW_DATA_DIR / "data_hep - data_hep.csv"
if not raw_csv_path.is_file():
    raw_csv_path = None

if raw_csv_path is None:
    print("Waiting for data/raw/data_hep - data_hep.csv.")
else:
    print(f"Dataset found: {raw_csv_path.name}")

## 4. Store the untouched CSV in SQLite

In [ ]:
raw_events = None
if raw_csv_path is not None:
    if table_exists("raw_events", DB_PATH):
        raw_events = read_table("raw_events", DB_PATH)
        print("Read the existing immutable raw_events table.")
    else:
        raw_events = load_raw_to_db(raw_csv_path, DB_PATH)
        print("Wrote the immutable raw_events table.")
    print(f"Loaded {len(raw_events):,} rows and {raw_events.shape[1]} columns.")
else:
    print("Ingestion skipped until exactly one source CSV is available.")

## 5. Audit the raw data

In [ ]:
raw_audit = audit_frame(raw_events) if raw_events is not None else None
if raw_audit is not None:
    print(f"Rows: {raw_audit['rows']:,}; columns: {raw_audit['columns']}")
    print(f"Duplicate rows: {raw_audit['duplicate_rows']:,}")
    print(f"Constant columns: {raw_audit['constant_columns']}")
    print(f"Six-way type counts: {raw_events['type'].value_counts().sort_index().to_dict()}")
    print(f"Missing cells: {sum(raw_audit['missing_counts'].values()):,}")
    print(f"Non-finite cells: {sum(raw_audit['non_finite_counts'].values()):,}")
    display(raw_events.head())
else:
    print("Audit skipped because no dataset has been loaded.")

## 6. Prepare the binary label

`prepare_professor_binary` applies the instructor's process mapping: 0 = charged B pair, 1 = neutral B pair, 2 = $c\bar c$, 3 = $u\bar u$, 4 = $d\bar d$, 5 = $s\bar s$. It combines 0/1 as signal (`target=1`) and 2/3/4/5 as continuum background (`target=0`). The original `type` and exported row number are not model inputs.

In [ ]:
prepared_events = prepare_professor_binary(raw_events) if raw_events is not None else None
if prepared_events is not None:
    print(f"Rows retained: {len(prepared_events):,}/{len(raw_events):,}")
    print("Excluded from prepared data: Unnamed: 0 and six-way type")
    display(prepared_events['target'].value_counts().sort_index().rename('rows'))
else:
    print("Binary preparation waits for the professor CSV.")

## 7. Store the 70/15/15 train/validation/test split

Split by row position, so duplicate pandas index labels cannot overlap assignments.

In [ ]:
events_with_split = None
if prepared_events is not None:
    if table_exists("cleaned_events", DB_PATH):
        events_with_split = read_table("cleaned_events", DB_PATH)
        print("Read persisted cleaned_events; did not re-split.")
    else:
        events_with_split = assign_split(
            prepared_events, "target",
            validation_size=VALIDATION_SIZE, test_size=TEST_SIZE,
        )
        write_table(events_with_split, "cleaned_events", db_path=DB_PATH)
        print("Wrote cleaned_events with one persisted split.")
    display(events_with_split.groupby(["split", "target"]).size().rename("rows"))
else:
    print("Split assignment waits for professor data.")

## 8. Training-only EDA and class balance

Explore training rows only; validation and test remain held out. Raw `type` and `Unnamed: 0` columns are rejected. Counts are unweighted.

In [ ]:
train_events = training_events(events_with_split) if events_with_split is not None else None
if train_events is not None:
    physics_features = feature_columns(train_events)
    print(f'Training rows: {len(train_events):,}; features: {len(physics_features)}')
    balance = class_balance(train_events)
    display(balance)
    display(plot_class_balance(balance))
else:
    print('EDA waits for the instructor dataset.')

## 9. Feature distributions and separation

Plot representative event-shape variables; rank all physics features by single-feature ROC-AUC. This ranking is not a final model result.

In [ ]:
separation = None
if train_events is not None:
    display(plot_feature_distributions(train_events,
           ['R2', 'thrust', 'thrustBm', 'foxWolframR2']))
    separation = feature_separation(train_events)
    display(separation.head(15))
else:
    print('Feature EDA waits for training rows.')

## 10. Correlations and R2 physics check

Check whether signal has lower R2 than continuum; investigate any reversal before modeling.

In [ ]:
if train_events is not None:
    display(plot_correlation_heatmap(feature_correlations(train_events)))
    r2_summary = separation.loc[separation['feature'] == 'R2'].iloc[0]
    print(f"R2 median: signal={r2_summary['signal_median']:.4f}, background={r2_summary['background_median']:.4f}")
    print('Expected R2 direction?', r2_summary['signal_median'] < r2_summary['background_median'])
else:
    print('Physics check waits for training rows.')

## 11. SQLite read, write, and modify

Store and update only the derived training EDA summary; raw and cleaned events stay unchanged.

In [ ]:
if separation is not None:
    eda_summary = separation.assign(status='computed')
    write_table(eda_summary, 'eda_feature_summary', if_exists='replace', db_path=DB_PATH)
    top_feature = eda_summary.iloc[0]['feature']
    changed = update_cell('eda_feature_summary', 'status', 'candidate_for_review',
                          'feature', top_feature, db_path=DB_PATH)
    print(f'Updated {changed} derived row(s): {top_feature}')
    display(read_table('eda_feature_summary', DB_PATH).head(10))
else:
    print('Derived SQLite summary waits for EDA.')

## 12. Training-defined baselines

Compare a training-majority prediction and a training-selected R2 cut on validation. Test rows remain untouched.

In [ ]:
baseline_definition, baseline_validation_results = (None, None)
if events_with_split is not None:
    baseline_definition, baseline_validation_results = evaluate_baselines(events_with_split)
    print(f'Training-selected R2 cut: {baseline_definition.r2_cut:.6f}')
    display(baseline_validation_results)
else:
    print('Baselines wait for the instructor data.')

## 13. Ankit's full-feature models

Train Logistic Regression and Decision Tree on stored training rows; show provisional validation metrics only. The final test split stays untouched.

In [ ]:
ankit_fitted_models, ankit_validation_results = (None, None)
if events_with_split is not None:
    ankit_fitted_models, ankit_validation_results = fit_ankit_validation(events_with_split)
    display(ankit_validation_results)
    print('Provisional validation metrics; final test rows are untouched.')
else:
    print('Modeling waits for the instructor data.')

## 14. Vishruti's full-feature models

Train a scaled Linear SVM and histogram Gradient Boosting BDT; compare all implemented methods on validation only.

In [ ]:
vishruti_fitted_models, vishruti_validation_results = (None, None)
if events_with_split is not None:
    vishruti_fitted_models, vishruti_validation_results = fit_vishruti_validation(events_with_split)
    provisional_validation_table = pd.concat(
        [baseline_validation_results, ankit_validation_results, vishruti_validation_results],
        ignore_index=True,
    )
    display(provisional_validation_table.sort_values('roc_auc', ascending=False))
    print('Validation-only comparison; final test rows remain untouched.')
else:
    print('Vishruti models wait for the instructor data.')

## Next: feature reduction and final team evaluation